In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [2]:
# Load CSV
try:
    df = pd.read_csv("../../Datasets/csv/UAVUnderNormalCyberattacks.csv", low_memory=False)
except FileNotFoundError:
    print("Error: The file '../Datasets/csv/UAVUnderNormalCyberattacks.csv' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [3]:
# Fill missing class labels according to dataset documentation
df['class_filled'] = df['class']
df.loc[0:9425, 'class_filled'] = 'Benign'
df.loc[13717:25388, 'class_filled'] = 'DoS Attack'
df.loc[26363:38369, 'class_filled'] = 'Replay Attack'
df.loc[39344:45027, 'class_filled'] = 'Evil Twin'
df.loc[50502:54783, 'class_filled'] = 'FDI'

# Remove header rows and NaNs, if any
df = df[df['class_filled'].notna()]
df = df[df['class_filled'] != 'class']

print("Unique labels after filling/cleaning:", df['class_filled'].unique())

Unique labels after filling/cleaning: ['Benign' 'DoS Attack' 'Replay Attack' 'Evil Twin' 'FDI']


In [4]:
# Constants
RANDOM_STATE = 1082
TEST_SIZE = 0.8

In [5]:
# Data Preprocessing

# Drop columns not used as features
drop_cols = ['timestamp_c', 'class', 'class_filled']
X = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

# Encode labels
le = LabelEncoder()
y = le.fit_transform(df['class_filled'])

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print("Unique classes after encoding:", np.unique(y))
num_classes = len(np.unique(y))

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

Label mapping: {'Benign': np.int64(0), 'DoS Attack': np.int64(1), 'Evil Twin': np.int64(2), 'FDI': np.int64(3), 'Replay Attack': np.int64(4)}
Unique classes after encoding: [0 1 2 3 4]


In [6]:
# Build and train XGBoost classifier
model = xgb.XGBClassifier(
    eval_metric='mae',
    random_state=RANDOM_STATE,
    objective='multi:softprob',
    num_class=num_classes
)

model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Model Accuracy on Test Set: 92.60%
Confusion Matrix:
[[7296   89    0    1  155]
 [ 406 8288    0    1  642]
 [   0    0 4546    1    0]
 [   0    0    0 3425    0]
 [ 473  782    0    1 8350]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.97      0.93      7541
           1       0.90      0.89      0.90      9337
           2       1.00      1.00      1.00      4547
           3       1.00      1.00      1.00      3425
           4       0.91      0.87      0.89      9606

    accuracy                           0.93     34456
   macro avg       0.94      0.94      0.94     34456
weighted avg       0.93      0.93      0.93     34456

